# 03b_Pre_Processing_Pipeline_Extension

Explainable AI Credit Risk Decision Platform : Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions

#### Reason:
This process is to clean, formats and transform the raw data into structured numerical representation so machine learning models can learn patterns accurately and efficiently.

In [68]:
# STANDARD LIBRARY
# ___________________________________________

from __future__ import annotations

import json
import logging
import warnings
import hashlib
import platform

from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

# DATA PROCESSING
# --------------------------------------

import numpy as np
import pandas as pd

# SCIPY
# --------------------------------------

from scipy import sparse
from scipy.sparse import load_npz

# VISUALISATION
# --------------------------------------

import matplotlib.pyplot as plt
import seaborn as sns

# REPRODUCIBILITY
# ---------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")

# DISPLAY CONFIGURATION
# ------------------------------------

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 180)

pd.set_option("display.max_colwidth", 120)

In [69]:
# PROJECT DIRECTORIES
# __________________________________________________

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

FEATURE_STORE_DIR = DATA_DIR / "feature_store"

REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR = PROJECT_ROOT / "models"

LOG_DIR = PROJECT_ROOT / "logs"

METADATA_DIR = REPORT_DIR / "metadata"

VALIDATION_DIR = REPORT_DIR / "validation"

FIGURE_DIR = REPORT_DIR / "figures"

In [70]:
# LOGGING CONFIGURATION
# _______________________________________________________

LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "application.log"

logging.basicConfig(level=logging.INFO,
                    
    format="%(asctime)s | %(levelname)s | %(message)s",
                    
    handlers=[logging.FileHandler(LOG_FILE)
              
              ,logging.StreamHandler()],force=True)

logger = logging.getLogger(__name__)

logger.info("Notebook Started")

2026-07-16 03:07:47,467 | INFO | Notebook Started


In [71]:
# UNIVERSAL DATASET LOADER
# _________________________________________________________

def load_dataset(path: Path):
    
    logger.info(f"Loading {path.name}")

    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    elif suffix == ".parquet":
        return pd.read_parquet(path)

    elif suffix == ".npz":
        return load_npz(path)

    else:
        raise ValueError(f"Unsupported file type: {suffix}")

In [72]:
# DATASET SUMMARY
# _________________________________________________________

def dataset_summary(df: pd.DataFrame, name: str):

    summary = {

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": int(df.isna().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum())}

    display(pd.DataFrame(summary, index=[0]))

    logger.info(summary)

    return summary

In [73]:
# DATA VALIDATION
# _____________________________________________________

def validate_dataframe(df: pd.DataFrame,
                       name: str,
                       key: str = None):
    report = {

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": int(df.isna().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum())}

    if key is not None:

        report["Missing Key"] = int(df[key].isna().sum())

        report["Unique Keys"] = df[key].nunique()

    report = pd.DataFrame(report, index=[0])

    return report

In [74]:
# Saving report
# _____________________________________________________

def save_report(df: pd.DataFrame,filename: str,
                
                folder: Path = REPORT_DIR):
    """
    Export DataFrame as CSV.
    """

    folder.mkdir(parents=True, exist_ok=True)

    output = folder / filename

    df.to_csv(output, index=False)

    logger.info(f"Saved: {output.name}")

In [133]:
# LOCATE DATASET
# # _____________________________________________________

LOAN_DATA_PATH = PROCESSED_DIR / "lendingclub_clean.csv"

logger.info(f"Dataset path: {LOAN_DATA_PATH}")

2026-07-16 11:37:35,882 | INFO | Dataset path: /Users/emmanuelahadzi/data/processed/lendingclub_clean.csv


In [134]:
# LOAD DATASET
# # _____________________________________________________

loan_df = load_dataset(LOAN_DATA_PATH)

logger.info("Processed Lending Club dataset loaded successfully.")

2026-07-16 11:37:36,810 | INFO | Loading lendingclub_clean.csv
2026-07-16 11:37:43,087 | INFO | Processed Lending Club dataset loaded successfully.


In [135]:
# DATA PREVIEW
# _____________________________________________________

display(loan_df.head())

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=68407277,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,NaN,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=68355089,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,NaN,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300

In [136]:
# SELECT APPROVED FEATURES
# _____________________________________________________

print("Select Approved Features")

# Features approved for Matrix D
# ---------------------------------------

APPROVED_FEATURES = [

    # Identifier
    "id",

    # Target
    "loan_status",

    # Time Key
    "issue_d",

    # Core Borrower Features
    "loan_amnt",
    "term",
    "grade",
    "int_rate",
    "annual_inc",
    "dti",
    "home_ownership",
    "title",
    "revol_util",

    # NLP Text
    "title_clean"]

# Keep only available features
selected_features = [feature
                     
                     for feature in APPROVED_FEATURES

    if feature in loan_df.columns]

removed_features = sorted(

    set(loan_df.columns) -

    set(selected_features))

loan_df = loan_df[selected_features].copy()

logger.info(f"Approved Features : {len(selected_features)}")

logger.info(f"Removed Features  : {len(removed_features)}")

print(f"Approved Features : {len(selected_features)}")
print(f"Removed Features  : {len(removed_features)}")

2026-07-16 11:37:43,298 | INFO | Approved Features : 12
2026-07-16 11:37:43,298 | INFO | Removed Features  : 139


Select Approved Features
Approved Features : 12
Removed Features  : 139


In [137]:
# HANDLE REMAINING MISSING VALUES
# # _____________________________________________________

print("Handle Remaining Missing Values")

numeric_columns = loan_df.select_dtypes(

    include=["number"]).columns

categorical_columns = loan_df.select_dtypes(

    include=["object", "category"]).columns

imputation_summary = []


# Numeric → Median
# -----------------------------------------------------

for column in numeric_columns:

    missing = loan_df[column].isna().sum()

    if missing > 0:

        loan_df[column] = loan_df[column].fillna(

            loan_df[column].median())

        imputation_summary.append([column,"Median",missing])


# Categorical → Mode
# ---------------------------------------------------

for column in categorical_columns:

    missing = loan_df[column].isna().sum()

    if missing > 0:

        loan_df[column] = loan_df[column].fillna(

            loan_df[column].mode()[0])

        imputation_summary.append([column,"Mode",missing])

imputation_summary = pd.DataFrame(

    imputation_summary,

    columns=["Feature","Imputation","Missing Values Filled"])

# Text Columns
# -----------------------------

if "title_clean" in loan_df.columns:

    loan_df["title_clean"] = loan_df["title_clean"].fillna("")

display(imputation_summary)

Handle Remaining Missing Values


,Feature,Imputation,Missing Values Filled
0,loan_amnt,Median,2
1,int_rate,Median,2
2,annual_inc,Median,2
3,dti,Median,218
4,revol_util,Median,267
5,loan_status,Mode,2
6,issue_d,Mode,2
7,term,Mode,2
8,grade,Mode,2
9,home_ownership,Mode,2


In [138]:
# STANDARDISE DATA TYPES
# _____________________________________________________

# Identifier
loan_df["id"] = loan_df["id"].astype("string")

# Dates
loan_df["issue_d"] = pd.to_datetime(loan_df["issue_d"],errors="coerce")

# Categories
categorical_cols = [

    "loan_status",
    "term",
    "grade",
    "home_ownership"]

for col in categorical_cols:

    loan_df[col] = loan_df[col].astype("category")

# Floats
numeric_cols = [

    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "revol_util"]

for col in numeric_cols:

    loan_df[col] = pd.to_numeric(
        loan_df[col],
        errors="coerce")

logger.info("Schema standardised.")

2026-07-16 11:37:46,858 | INFO | Schema standardised.


In [139]:
# VALIDATE DATASET
# _____________________________________________________

print("Validate Dataset")

loan_validation = validate_dataframe(df=loan_df,
                                     
                                     name="Processed Lending Club Dataset")

display(loan_validation)

# Additional validation

remaining_missing = loan_df.isna().sum().sum()

duplicate_rows = loan_df.duplicated().sum()

print(f"Remaining Missing Values : {remaining_missing:,}")

print(f"Duplicate Rows           : {duplicate_rows:,}")

logger.info("Dataset validation completed.")

Validate Dataset


,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Processed Lending Club Dataset,500000,12,0,0


2026-07-16 11:37:49,564 | INFO | Dataset validation completed.


Remaining Missing Values : 0
Duplicate Rows           : 0


In [140]:
# DATASET SUMMARY
# _____________________________________________________

print("Dataset Summary")

loan_summary = pd.DataFrame({

    "Metric":[

        "Rows",

        "Columns",

        "Numeric Features",

        "Categorical Features",

        "Memory (MB)"],

    "Value":[

        loan_df.shape[0],

        loan_df.shape[1],

        loan_df.select_dtypes(include="number").shape[1],

        loan_df.select_dtypes(exclude="number").shape[1],

        round(

            loan_df.memory_usage(deep=True).sum()/1024**2,2)]})

display(loan_summary)

dataset_summary(loan_df,"Processed Lending Club Dataset")

Dataset Summary


,Metric,Value
0,Rows,500000.00
1,Columns,12.00
2,Numeric Features,5.00
3,Categorical Features,7.00
4,Memory (MB),84.04


,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Processed Lending Club Dataset,500000,12,0,0


2026-07-16 11:37:54,683 | INFO | {'Dataset': 'Processed Lending Club Dataset', 'Rows': 500000, 'Columns': 12, 'Missing Values': 0, 'Duplicate Rows': 0}


{'Dataset': 'Processed Lending Club Dataset',
 'Rows': 500000,
 'Columns': 12,
 'Missing Values': 0,
 'Duplicate Rows': 0}

In [141]:
# EXPORT REPORTS
# _____________________________________________________

print("Export Reports")

feature_inventory = pd.DataFrame({

    "Feature": loan_df.columns,

    "Data Type": loan_df.dtypes.astype(str)})

save_report(validation_report,"loan_validation_report.csv")

save_report(loan_summary,"loan_dataset_summary.csv")

save_report(feature_inventory,"processed_lendingfeature_inventory.csv")

removed_feature_report = pd.DataFrame({

    "Removed Feature": removed_features})

save_report(removed_feature_report,"removed_features.csv")

logger.info("Section 3 completed successfully.")

logger.info("SECTION 3 COMPLETED")

print("\n✓ Processed Lending Club Dataset successfully prepared for Matrix D.")

2026-07-16 11:37:58,958 | INFO | Saved: loan_validation_report.csv
2026-07-16 11:37:58,960 | INFO | Saved: loan_dataset_summary.csv
2026-07-16 11:37:58,962 | INFO | Saved: processed_lendingfeature_inventory.csv
2026-07-16 11:37:58,964 | INFO | Saved: removed_features.csv
2026-07-16 11:37:58,965 | INFO | Section 3 completed successfully.
2026-07-16 11:37:58,965 | INFO | SECTION 3 COMPLETED


Export Reports

✓ Processed Lending Club Dataset successfully prepared for Matrix D.


In [145]:
# SAVE CLEAN DATASET
# _____________________________________________________

CLEAN_LOAN_PATH = PROCESSED_DIR / "clean_processed_lending_club.parquet"

loan_df.to_parquet(CLEAN_LOAN_PATH,engine="pyarrow",index=False)

logger.info("Clean Lending Club dataset saved.")

2026-07-16 11:47:07,448 | INFO | Clean Lending Club dataset saved.


In [146]:
# Verify saved data
# _____________________________________________________

df_check = pd.read_parquet(CLEAN_LOAN_PATH)

print(df_check.shape)

print(df_check.isna().sum().sum())

print(df_check.dtypes)

(500000, 12)
0
id                string[python]
loan_status             category
issue_d           datetime64[ns]
loan_amnt                float64
term                    category
grade                   category
int_rate                 float64
annual_inc               float64
dti                      float64
home_ownership          category
title                     object
revol_util               float64
dtype: object
